In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'tensorflow', 'keras', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'keras': 'keras==3.14.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'tensorflow' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'tensorflow' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "061a82116be00c585cab41d5b2a790e2a4f96d44"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/061a82116be00c585cab41d5b2a790e2a4f96d44/d2l"
for name in ('__init__.py', 'tensorflow.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# ConvNeXt: Modernizing a ResNet Step by Step

Every network of this chapter so far is a convolutional network, and until 2020
the best image classifiers were convolutional networks too. In 2021 a network
without convolutions overtook them. The Swin Transformer [@liu2021swin],
built around a layer called *attention*, reached 81.3% top-1 accuracy on
ImageNet, against 76.1% for a ResNet-50 of similar cost. Did attention cause
the gap? If it did, convolutional networks were obsolete.

A sketch of the newcomer is all we need here; that section
treats it properly. A *vision transformer*
[@Dosovitskiy.Beyer.Kolesnikov.ea.2021] cuts the image into a grid of
patches, turns each patch into a vector, and passes these vectors through a
stack of identical blocks. Each block has two parts. An attention layer lets
every patch take a weighted average of the features of other patches, with
weights computed from the features themselves; a small MLP then transforms each
patch's vector on its own. Swin computes attention within local $7 \times 7$
windows of patches and lowers the resolution between stages as a convnet does.

The previous section suggests caution. Swin differs from a 2015 ResNet in many
ways besides attention: it was trained with a newer recipe, its stem slices the
image into patches, and it uses another activation, another normalization, and
other stage depths. Which of these differences carried the improvement?
@liu2022convnet answered by experiment. Starting from a ResNet-50, they
applied one Swin-inspired change at a time, measured ImageNet accuracy after
each, and never added attention. The result, *ConvNeXt*, reaches 82.0%. Its
ingredients come from earlier sections: the training recipe of
that section, the depthwise convolutions and inverted
bottlenecks of that section, and the layer normalization of
that section. Every row of their table is a full ImageNet training
run, so we cannot repeat the experiment. We read the table instead and build the
network it ends in. Then we run a small roadmap of our own on Fashion-MNIST:
starting from a ResNet-18 with the same number of parameters, we apply the
paper's changes in three steps and train after each.

In [ ]:
import math
import numpy as np
import tensorflow as tf
from d2l import tensorflow as d2l

## The Modernization Roadmap

the table lists the changes in the order the paper
applied them. Each row keeps every change above it, so the accuracies are
cumulative, and the computational cost stays near that of Swin-T, about
4.5 billion multiply-adds per image (the paper's 4.5 GFLOPs).

<!-- tbl-caption:The ConvNeXt modernization roadmap [@liu2022convnet]. ImageNet top-1 accuracy of a ResNet-50 as design changes accumulate; each row includes all rows above it, at roughly constant cost. Swin-T, a transformer of similar cost, reaches 81.3%. -->

| Change | What it means | Top-1 |
|:--|:--|:--|
| (starting point) | ResNet-50 with its 2015 recipe | 76.1% |
| Modern training recipe | 300 epochs, AdamW, Mixup, label smoothing, stochastic depth (that section) | 78.8% |
| Stage ratio | blocks per stage $(3, 4, 6, 3) \to (3, 3, 9, 3)$ | 79.4% |
| Patchify stem | $7 \times 7$ stride-2 convolution plus pooling $\to$ one $4 \times 4$ stride-4 convolution | 79.5% |
| Depthwise convolutions | the $3 \times 3$ convolution becomes depthwise; width grows from 64 to 96 channels | 80.5% |
| Inverted bottleneck | expand $4\times$ inside the block instead of compressing | 80.6% |
| Large kernels | depthwise convolution moved to the front and enlarged to $7 \times 7$ | 80.6% |
| ReLU $\to$ GELU | a smooth activation, as in transformers | 80.6% |
| Fewer activations | one activation per block instead of three | 81.3% |
| Fewer normalizations | one normalization per block instead of three | 81.4% |
| BN $\to$ LN | layer normalization over channels at each position | 81.5% |
| Separate downsampling | an LN plus $2 \times 2$ stride-2 convolution between stages | 82.0% |

The first change gives the largest single improvement, and it does not touch
the architecture at all. This is the recipe effect of
that section, here worth 2.7 points on an unchanged
ResNet-50. Of the 5.9 points that separate the 2015 ResNet-50 from ConvNeXt,
2.7 thus come from training and 3.2 from the architecture, at least in this
order of application. Papers that compared a well-tuned transformer against a
ResNet trained the 2015 way were therefore comparing recipes as much as
architectures.

### Macro Design

The next two rows change the shape of the network as a whole. Swin-T
distributes its blocks across four stages in the ratio $1{:}1{:}3{:}1$,
spending most of its depth where the feature maps are small and each block is
cheap. Adopting the same ratio turns ResNet-50's $(3, 4, 6, 3)$ blocks into
$(3, 3, 9, 3)$ and adds 0.6 points. The stem changes more. ResNet begins with a
$7 \times 7$ stride-2 convolution followed by max-pooling, a fourfold reduction
computed with overlapping windows. A vision transformer instead slices the
image into non-overlapping patches and multiplies the pixels of each patch by
one shared matrix. That is nothing more exotic than a convolution whose kernel
size equals its stride (that section), here a $4 \times 4$ convolution
with stride 4. This *patchify* stem replaces the ResNet stem with no loss
(79.5%) and produces one feature vector per $4 \times 4$ patch, the same object
a transformer computes.

### Depthwise Convolutions and the Inverted Bottleneck

A transformer block separates two kinds of mixing. Attention combines
information across positions, and the MLP that follows combines the channels of
each position on its own. A convolutional network can separate them the same
way, with the two layers of that section: a *depthwise*
convolution mixes positions but not channels, and a $1 \times 1$ convolution
mixes channels at each position but not positions. ConvNeXt makes the
$3 \times 3$ convolution of the ResNet bottleneck depthwise. That cuts its cost
so much that the network can afford more width: 96 channels in the first stage
instead of 64, matching Swin-T, which brings 80.5%.

Why does the ResNet bottleneck compress its input before the spatial
convolution and expand it afterwards? Because the cost of a dense convolution
grows with the square of the width. The cost of a depthwise convolution grows
only in proportion to the width, so the reason to compress is gone, and the
block can keep a thin stream between blocks and expand it inside, as
MobileNetV2 does (that section). The transformer's MLP has the
same shape, a $1 \times 1$ expansion by $4\times$, an activation, and a
projection back. *Inverting* the bottleneck this way adds a little (80.6%).

With the dense convolution gone, the kernel size becomes cheap to change. The
depthwise convolution moves to the front of the block, so that it acts on the
thin stream of $c$ channels instead of the $4c$ expanded ones. This alone costs
0.7 points (to 79.9%), the only step backwards on the roadmap, and enlarging
the kernel from $3 \times 3$ to $7 \times 7$ recovers them. A $7 \times 7$
kernel widens the receptive field as much as three stacked $3 \times 3$ layers
(that section). VGG chose the stack (that section) because dense
kernels cost $k^2 c_\textrm{i} c_\textrm{o}$ multiply-adds per position. A
depthwise kernel costs $k^2 c$, so going from $3 \times 3$ to $7 \times 7$
multiplies a small term by $49/9 \approx 5$. The authors report no further gain
from kernels beyond $7 \times 7$, which is also the window size of Swin's
attention.

### Micro Design

The remaining rows change individual layers. GELU
[@Hendrycks.Gimpel.2016], the smooth variant of the ReLU that transformers
use, changes nothing by itself (80.6%). What matters is *how many* activations
and normalizations the block contains. A ResNet bottleneck follows every
convolution with a normalization and an activation: three of each. A
transformer block has one activation, inside its MLP, and one normalization in
front of each of its two parts. Keeping only the GELU between the two
$1 \times 1$ convolutions adds 0.7 points, more than any other change in this
group. Why should a network improve when it loses two of its three
nonlinearities? We know of no convincing argument, and the gain is best
taken as an empirical finding. Keeping only one normalization adds 0.1, and
making that one a layer normalization over the channels at each spatial
position (that section), instead of a batch
normalization, adds another tenth (81.5%).
that section introduced normalization as an aid to optimization;
here fewer such layers, one per block, train better than three.
the figure shows the block before and after; its
layer scale $\boldsymbol{\gamma}$ is defined with the implementation below.

![The ResNet-50 bottleneck block and the ConvNeXt block that the roadmap turns it into. Three normalizations and three activations become one of each; the compress-process-expand bottleneck becomes an inverted bottleneck led by a $7 \times 7$ depthwise convolution, and a learnable per-channel layer scale $\boldsymbol{\gamma}$ multiplies the branch before the addition.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/arch-resnet-vs-convnext-block.svg)

Finally, ResNet downsamples inside the first block of each stage by giving its
$3 \times 3$ convolution a stride of 2. Swin uses a separate patch-merging
layer between stages. Adopting the same separation completes the roadmap at
82.0%: a layer normalization followed by a $2 \times 2$ stride-2 convolution
between stages, plus a normalization after the stem and one before the
classifier, without which training was unstable. the figure shows
the assembled ConvNeXt-T, with four stages of $(3, 3, 9, 3)$ blocks at widths
$(96, 192, 384, 768)$, in the stem-body-head layout this chapter has used since
that section.

![ConvNeXt-T. A patchify stem replaces convolution-plus-pooling; stages of 3, 3, 9, and 3 blocks are joined by separate LN plus $2 \times 2$ stride-2 downsampling layers; there is no pooling anywhere in the body.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/arch-convnext.svg)

At equal cost, ConvNeXt-T matches Swin-T without a single attention layer. The
gap of 2021 came from the training recipe and from design choices other than
attention. The experiment does not show that convolutions are better: under
comparable tuning the two families reach similar accuracy.

## Implementation

The roadmap removed more than it added, so the block is short.

### The ConvNeXt Block

The block applies, in order, a $7 \times 7$ depthwise convolution, a layer
normalization, a $1 \times 1$ convolution that expands the channels $4\times$,
a GELU, and a $1 \times 1$ convolution that projects back, and it adds the
result to its input. Two details of the paper's training setup come with it.
*Layer scale* [@touvron2021cait] multiplies the branch by a learnable
vector $\boldsymbol{\gamma}$, one entry per channel, initialized to $10^{-6}$.
(It shares its letter with the scale of batch normalization but is a
parameter of its own.) Every block then starts close to the identity, and the
network begins training as a shallow function. *Stochastic depth* drops the whole branch of a block at
random during training; we reuse `d2l.drop_path` of
that section.

The layer normalization acts on the channel vector at each spatial position. In
a channels-last layout (examples, height, width, channels) this is the default
layer normalization over the last axis, and a $1 \times 1$ convolution is a
linear layer applied to that axis. Following the paper, every convolution and
linear layer starts with weights drawn from a normal distribution with standard
deviation 0.02 and zero biases.
<!-- d2l:prose id=convnext-md-3 fw=tensorflow, jax -->

Images are stored channels-last, so the block uses the library's layer
normalization and dense layers directly.

In [ ]:
def normal_002():  # Keras needs a fresh initializer object per layer
    return tf.keras.initializers.RandomNormal(stddev=0.02)

def layer_norm():
    return tf.keras.layers.LayerNormalization(epsilon=1e-6)

class ConvNeXtBlock(tf.keras.layers.Layer):
    """Depthwise 7x7, norm, 1x1 expand, GELU, 1x1 project, scaled residual."""
    def __init__(self, dim, drop_prob=0.0, layer_scale=1e-6,
                 norm=layer_norm):
        super().__init__()
        self.dwconv = tf.keras.layers.DepthwiseConv2D(
            kernel_size=7, padding='same',
            depthwise_initializer=normal_002())
        self.norm = norm()
        self.pwconv1 = tf.keras.layers.Dense(
            4 * dim, kernel_initializer=normal_002())
        self.pwconv2 = tf.keras.layers.Dense(
            dim, kernel_initializer=normal_002())
        self.gamma = self.add_weight(
            shape=(dim,),
            initializer=tf.keras.initializers.Constant(layer_scale))
        self.drop_prob = drop_prob

    def call(self, X, training=False):
        Y = self.norm(self.dwconv(X), training=training)
        Y = self.pwconv2(tf.keras.activations.gelu(self.pwconv1(Y)))
        return X + d2l.drop_path(self.gamma * Y, self.drop_prob, training)

### The Full Network

The network is once more the loop over a tuple of (depth, channels) pairs of
that section. The stem is the patchify convolution plus a layer
normalization, and the downsampling layer sits between the stages; there is no
pooling in the body. Following the paper, the stochastic-depth probability
grows linearly from zero at the first block to its maximum at the last. For
Fashion-MNIST we use the widths $(40, 80, 160, 320)$ and depths $(2, 2, 6, 2)$
of ConvNeXt-atto, the smallest variant in common use; the third stage is the
deepest, as in ConvNeXt-T. The normalization layer is an argument, `norm`, and
the stem a method, `stem`, so that the experiment at the end of this section
can exchange them.

In [ ]:
class ConvNeXt(d2l.Classifier):
    def __init__(self, arch=((2, 40), (2, 80), (6, 160), (2, 320)),
                 lr=2e-3, num_classes=10, drop_path_max=0.1,
                 norm=layer_norm):
        super().__init__()
        self.save_hyperparameters()
        depths = [d for d, c in arch]
        rates = tf.linspace(0.0, drop_path_max, sum(depths)).numpy().tolist()
        self.net = tf.keras.models.Sequential(self.stem(arch[0][1], norm))
        b = 0
        for i, (depth, c) in enumerate(arch):
            if i > 0:  # Separate downsampling layer between stages
                self.net.add(norm())
                self.net.add(tf.keras.layers.Conv2D(
                    c, kernel_size=2, strides=2,
                    kernel_initializer=normal_002()))
            for _ in range(depth):
                self.net.add(ConvNeXtBlock(c, rates[b], norm=norm))
                b += 1
        self.net.add(tf.keras.layers.GlobalAvgPool2D())
        self.net.add(norm())
        self.net.add(tf.keras.layers.Dense(num_classes,
                                           kernel_initializer=normal_002()))

    def stem(self, c, norm):  # Patchify: a 4x4 convolution with stride 4
        return [tf.keras.layers.Conv2D(c, kernel_size=4, strides=4,
                                       kernel_initializer=normal_002()),
                norm()]

We train on Fashion-MNIST upsampled to $64 \times 64$. Such an input leaves
the stem as a $16 \times 16$ map, and the three downsampling layers reduce
it to $8 \times 8$, $4 \times 4$, and finally $2 \times 2$ before the head
pools it away. We check the output shape and count the parameters.

In [ ]:
model = ConvNeXt()
assert model(tf.random.normal((1, 64, 64, 1))).shape == (1, 10)
d2l.num_params(model)

The network has 3,376,450 parameters, about a third of the 11.2 million of
the ResNet-18 that that section trained on Fashion-MNIST. Because a
single wrongly sized layer changes it, the count is a sensitive check of any
reimplementation, ours included.

## A Small Roadmap on Fashion-MNIST

Every choice on the roadmap was made on 1.3 million ImageNet images at
$224 \times 224$. Which of them also help on small grayscale images? Nothing
guarantees that any do. We test them the way the paper found them: start from a
control, apply the changes cumulatively, and train after each step. The control
has as many parameters as our ConvNeXt: a ResNet-18 whose base width is reduced
from 64 to 35 channels. Every network uses the compact modern recipe B of
that section with its settings: AdamW, that is, Adam with
decoupled weight decay (that section), warmup and cosine decay,
label smoothing, and Mixup. Only the length differs: 20 epochs on the full
training set instead of 45 epochs on 10,000 images, with a warmup over the
first 10% of the steps instead of three epochs. Like recipe B, we leave out
stochastic depth, by setting ConvNeXt's rate to zero, so that the networks
differ in architecture alone.

The class `ModernRecipe` below packages that recipe as a mixin, so that one
copy serves every network: any classifier that stores its learning rate `lr`
and its total number of optimization steps `num_steps` can inherit the
optimizer, its schedule, and the training step. The validation loss stays the
plain cross-entropy of `Classifier`, so it means the same in every section.

In [ ]:
class ModernRecipe:
    """AdamW, linear warmup then cosine decay, label smoothing, Mixup."""
    def configure_optimizers(self):
        warmup = self.num_steps // 10
        schedule = lambda: self.lr * d2l.cosine_warmup(
            optim.iterations, self.num_steps, warmup)
        optim = tf.keras.optimizers.AdamW(schedule, weight_decay=0.05,
                                          epsilon=1e-8)
        return optim

    def training_step(self, batch):
        X, y_a, y_b, lam = d2l.mixup(*batch, alpha=0.2)
        Y_hat = self(X, training=True)
        loss = tf.keras.losses.CategoricalCrossentropy(
            from_logits=True, label_smoothing=0.1)
        one_hot = lambda y: tf.one_hot(y, Y_hat.shape[1])
        return (lam * loss(one_hot(y_a), Y_hat)
                + (1 - lam) * loss(one_hot(y_b), Y_hat))

class ModernConvNeXt(ModernRecipe, ConvNeXt):
    def __init__(self, num_steps, lr=2e-3, norm=layer_norm):
        super().__init__(lr=lr, drop_path_max=0.0, norm=norm)
        self.num_steps = num_steps

The control stacks the `Residual` blocks of that section in the
layout of ResNet-18: a $7 \times 7$ stride-2 stem with max-pooling, four
stages of two blocks each with 35, 70, 140, and 280 channels, and a
global-average-pooling head.

In [ ]:
def he_init():  # He initialization; Keras needs one instance per layer
    return tf.keras.initializers.VarianceScaling(2.0, 'fan_in',
                                                 'untruncated_normal')

class CompactResNet18(ModernRecipe, d2l.Classifier):
    """ResNet-18 with base width 35 instead of 64, trained by the recipe."""
    def __init__(self, num_steps, lr=2e-3, num_classes=10, base=35):
        super().__init__()
        self.save_hyperparameters()
        self.net = tf.keras.models.Sequential([
            tf.keras.layers.ZeroPadding2D(3),  # Pad every side
            tf.keras.layers.Conv2D(base, kernel_size=7, strides=2,
                                   use_bias=False,
                                   kernel_initializer=he_init()),
            tf.keras.layers.BatchNormalization(momentum=0.9, epsilon=1e-5),
            tf.keras.layers.Activation('relu'),
            tf.keras.layers.ZeroPadding2D(1),  # Inputs are >= 0 after ReLU
            tf.keras.layers.MaxPool2D(pool_size=3, strides=2)])
        for i, c in enumerate((base, 2 * base, 4 * base, 8 * base)):
            for j in range(2):
                down = i > 0 and j == 0
                self.net.add(d2l.Residual(c, use_1x1conv=down,
                                          strides=2 if down else 1))
        self.net.add(tf.keras.layers.GlobalAvgPool2D())
        self.net.add(tf.keras.layers.Dense(num_classes,
                                           kernel_initializer=he_init()))

Our roadmap groups the paper's changes by the part of the network they act on.
The first step replaces the blocks and stages of the ResNet by those of
ConvNeXt (the $7 \times 7$ depthwise convolution in an inverted bottleneck,
one GELU and one normalization per block, layer scale, the $(2, 2, 6, 2)$
stages, and the separate downsampling layers) but keeps the ResNet stem and
batch normalization; it covers eight rows of the table. The
second step replaces batch normalization by layer normalization. The third
replaces the ResNet stem by the patchify stem, which completes our ConvNeXt.
The paper changed the stem before the blocks; the final network does not
depend on the order, although the gain credited to each step can. In code,
the steps are the argument `norm` and a subclass, `ResNetStemConvNeXt`, that
overrides the method `stem`.

In [ ]:
def batch_norm():  # Decay 0.9 and epsilon 1e-5, as in the ResNet
    return tf.keras.layers.BatchNormalization(momentum=0.9, epsilon=1e-5)

class ResNetStemConvNeXt(ModernConvNeXt):
    """ConvNeXt with the stem of ResNet-18 instead of the patchify stem."""
    def stem(self, c, norm):  # 7x7 convolution with stride 2, max-pooling
        return [tf.keras.layers.ZeroPadding2D(3),  # Pad every side
                tf.keras.layers.Conv2D(c, kernel_size=7, strides=2,
                                       kernel_initializer=normal_002()),
                norm(), tf.keras.layers.Activation('relu'),
                tf.keras.layers.ZeroPadding2D(1),  # Inputs >= 0 after ReLU
                tf.keras.layers.MaxPool2D(pool_size=3, strides=2)]

roadmap = {'compact ResNet-18': CompactResNet18,
           'ConvNeXt blocks and stages, BN': (
               lambda num_steps: ResNetStemConvNeXt(num_steps,
                                                    norm=batch_norm)),
           '+ BN -> LN': ResNetStemConvNeXt,
           '+ patchify stem = ConvNeXt': ModernConvNeXt}

We train the four networks in turn and record the parameters and the
validation accuracy after the last epoch of each.

In [ ]:
data = d2l.FashionMNIST(batch_size=128, resize=(64, 64))
num_steps = 20 * len(data.train_dataloader())
results = {}
for name, make_model in roadmap.items():
    trainer = d2l.Trainer(max_epochs=20)
    trainer.board.display = False  # Four runs; we tabulate the results
    with d2l.try_gpu():
        model = make_model(num_steps)
        trainer.fit(model, data)
    results[name] = (d2l.num_params(model), trainer.history['val_acc'][-1])

In [ ]:
for name, (params, accuracy) in results.items():
    print(f'{name:34s} {params:>9,} parameters, '
          f'validation accuracy {accuracy:.3f}')

<!-- tbl-caption:Our roadmap on Fashion-MNIST beside the paper's. The last column is the validation accuracy after 20 epochs of the modern recipe at $64 \times 64$, rounded to the stability of repeated runs; the paper's column adds up the ImageNet top-1 gains of the rows of the table that each step covers. -->

| Step | Rows of the paper | Gain on ImageNet | Parameters | Validation accuracy |
|:--|:--|--:|--:|--:|
| Compact ResNet-18 | (ResNet-50 with the modern recipe) | | 3.35M | ≈ 94% |
| ConvNeXt blocks and stages, BN | stage ratio to fewer normalizations, separate downsampling | +3.0 | 3.38M | ≈ 94% |
| BN $\to$ LN | BN $\to$ LN | +0.1 | 3.38M | ≈ 94% |
| Patchify stem (our ConvNeXt) | patchify stem | +0.1 | 3.38M | ≈ 91.5% |

The changes that make up the ConvNeXt block transfer. With the ResNet stem,
the ConvNeXt is as accurate as the ResNet-18, with batch or with layer
normalization: the first three rows agree within the spread of repeated
runs, a few tenths of a point. The eight rows that were worth three points
on ImageNet gain nothing here, and they lose nothing either. All of the
loss, about two points, comes from the one step that was neutral on
ImageNet, the patchify stem. The stem it replaces convolves at twice the
final resolution and then max-pools over neighboring positions; the patchify
stem reaches the $16 \times 16$ grid in a single linear step, without
pooling. Which of these differences matters we have not tested.

The control is fair as far as arithmetic goes: counting the multiply-adds of
every convolution and linear layer for one $64 \times 64$ image gives 43.2
million for the ResNet, 46.2 million for the ConvNeXt with the ResNet stem,
and 44.3 million for our ConvNeXt. We did not match the running time, which
multiply-adds do not determine (that section), and 20 epochs
are short for a design tuned for 300. Within these limits the experiment
locates the two points that our ConvNeXt loses. Design choices selected on
ImageNet do not transfer automatically, but here only one of them fails to.

## Limits and Open Problems

ConvNeXt answers one question, which design choices closed the gap of 2021, and
leaves two open. The first is pretraining. Large vision models are now often
pretrained without labels, for instance as *masked autoencoders*
[@he2022masked] that hide most patches of an image and learn to
reconstruct them. That is natural for a network that treats the image as a set
of patches and awkward for a convolution that slides across the holes. ConvNeXt
V2 [@woo2023convnextv2] makes it work; problem
5 implements the layer it
adds. The second is how far a convolution needs to see. A $7 \times 7$ kernel
sees a small window, whereas global attention lets every position read from
every other. Whether much larger kernels [@ding2022replknet], convolutions
that learn where to sample, or attention is the better way to see the whole
image at a given cost is still open.

## Summary

ConvNeXt is a controlled experiment wearing an architecture's name. Starting
from a ResNet-50, it applied the design choices of the Swin Transformer one at
a time and kept those that helped: a modern training recipe, a $1{:}1{:}3{:}1$
stage ratio, a patchify stem, depthwise convolutions in an inverted bottleneck
led by a $7 \times 7$ kernel, one GELU and one layer normalization per block,
and separate downsampling layers. The result matches Swin-T at equal cost
without attention, and almost half of its gain over the 2015 ResNet-50 comes
from the recipe alone. Its block separates mixing across positions from mixing
across channels, as a transformer block separates attention from its MLP.
Choices selected at ImageNet scale need not transfer, and a small roadmap
shows which do not: on upsampled Fashion-MNIST the ConvNeXt block, its stages,
and layer normalization match a ResNet-18 with the same number of parameters,
and the patchify stem alone costs about two points.

## Exercises

1. **Roadmap ablations.** The roadmap in the table
   applies its changes in one fixed order, and each row's gain is measured
   with every earlier change in place.
    1. Just before the "Large kernels" row, each block expands its residual
       width $c$ to $4c$ with a $1 \times 1$ convolution, applies a
       $3 \times 3$ depthwise convolution to the $4c$ channels, and projects
       back to $c$ with a second $1 \times 1$ convolution. The row first
       moves the depthwise convolution in front of the expansion, where it
       acts on $c$ channels, and then enlarges its kernel to $7 \times 7$.
       Count the multiply-adds per spatial position of the depthwise
       convolution before the move, after the move, and after the
       enlargement. Compare the net change over the row with the change
       that enlarging the kernel to $7 \times 7$ in place, on the $4c$
       channels, would have caused.
    1. Choose two rows whose gains you expect to depend on their order, and
       design the additional runs that would measure the interaction.

1. **Where the parameters live.**
    1. For the `ConvNeXt` of this section, compute the fraction of
       parameters in the depthwise convolutions, in the $1 \times 1$
       expansions and projections, in the normalizations and layer-scale
       vectors inside the blocks, in the downsampling layers, and in the
       stem and head. Check that the total is 3,376,450.
    1. Express the parameter count of a `ConvNeXtBlock` and of a `Residual`
       block of that section without the $1 \times 1$ shortcut as
       functions of the width $c$. Which design decision makes the ConvNeXt
       block cheaper, and by what factor for large $c$?

1. **Normalization axes.** The activations entering a normalization layer
   form a minibatch of $n$ images, each with $c$ channels on an
   $h \times w$ map. Consider three layers: batch normalization, the
   per-position layer normalization of `ConvNeXtBlock`, and a layer
   normalization over all $chw$ values of an image with one learnable scale
   and one learnable shift per value.
    1. For each layer, state which values share one mean and variance, and
       count its learnable parameters.
    1. A network trained on $64 \times 64$ images is evaluated on
       $224 \times 224$ images. Which of the three layers can be applied to
       the larger images with their trained parameters unchanged? Explain.

1. [code] **Layer scale.** Under the recipe of the roadmap in
   the table, train `ConvNeXt` with $\boldsymbol{\gamma}$
   initialized to $10^{-6}$ (the default), with $\boldsymbol{\gamma}$
   initialized to $1$, and with layer scale removed, and compare the
   training curves. Layer scale at $10^{-6}$ and stochastic depth
   (the equation of that section) both
   make the network effectively shallower than its block count. For each
   mechanism, explain how it does so, state when during training it acts,
   and state whether it changes the network used at evaluation.

1. [code] **Global Response Normalization.** ConvNeXt V2
   [@woo2023convnextv2] inserts GRN after the GELU of the block and
   removes layer scale. For a feature map $\mathbf{X}$ with $C$ channels,
   let $g_c$ be the $\ell_2$ norm of channel $c$ over all spatial positions
   and $\bar{g}$ the mean of $g_1, \ldots, g_C$. GRN rescales each channel
   by its normalized response and adds a learnable per-channel affine term
   and a residual,
   $$\mathrm{GRN}(\mathbf{X})_c = \gamma_c \frac{g_c}{\bar{g} + \epsilon} \mathbf{X}_c + \beta_c + \mathbf{X}_c,$$
   with a small $\epsilon$ against division by zero ($10^{-6}$ in the
   paper's code), and with $\gamma_c$ and $\beta_c$ initialized to zero so
   that the layer starts as the identity.
    1. Implement GRN in about five lines and insert it into `ConvNeXtBlock`
       after the GELU, dropping layer scale.
    1. Retrain under the recipe of the roadmap in
       the table and compare
       the validation accuracy.

1. [code] **Depthwise kernel size.** ● Under the recipe of the roadmap in
   the table, train `ConvNeXt` with $3 \times 3$ and
   with $11 \times 11$ depthwise convolutions that leave the feature-map
   size unchanged, and compare accuracy, parameter count, and time per
   epoch against the $7 \times 7$ baseline. Does accuracy saturate at
   $7 \times 7$ here as @liu2022convnet report on ImageNet? How
   does the change in time per epoch compare with the change in the
   depthwise term's arithmetic, and why?

1. [code] **Channel redundancy.** ● Take the `ConvNeXt` trained with GRN in
   problem 5 and the one
   trained without it under the same recipe. Measure the channel redundancy
   that GRN was designed to prevent, at the input of the second $1 \times 1$
   convolution of every block. For each validation image, flatten each
   channel's map into a vector over the spatial positions, and average the
   cosine similarity over all pairs of channels. Average over the images
   and compare the networks with and without GRN block by block. In which
   stages, if any, does GRN make the channels less similar?

1. [code] **Recipe downgrade.** ● The first and last rows of
   the table train two architectures under the modern
   recipe. Complete the ablation square by training both models under
   recipe A of that section: SGD with momentum, step
   decay, and neither Mixup nor label smoothing, with the learning rate
   retuned. How much of each network's accuracy survives the downgrade, and
   does the recipe help both architectures equally? Compare the size of the
   effect with the 2.7 points that the "Modern training recipe" row of
   the table attributes to the recipe on ImageNet.

[Discussions](https://d2l.discourse.group/t/36762)